In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

data_dir = Path('/content/battery_data/SL_Dataset_SECL_INR21700-M50T')
labels = pd.read_csv('/content/drive/MyDrive/battery_data/stanford_soh_labels.csv')

In [ ]:
def read_channel_sheet(xlsx_path):
    xl = pd.ExcelFile(xlsx_path)
    channel_sheets = [s for s in xl.sheet_names if s.startswith('Channel_')]
    if not channel_sheets:
        return None
    return pd.read_excel(xlsx_path, sheet_name=channel_sheets[0])


def parse_cell_id(path):
    text = str(path).lower()
    match = re.search(r'_(g1|v4|v5|w8|w9|w10)_', text)
    return match.group(1).upper() if match else None


def parse_cycling_id(path):
    text = str(path)
    match = re.search(r'Cycling_(\d+)', text)
    return int(match.group(1)) if match else None


def find_temperature_cols(df):
    return [
        c for c in df.columns
        if 'temp' in str(c).lower() or 'temperature' in str(c).lower()
    ]

In [ ]:
def extract_post_charge_features(df):
    required = ['Test_Time(s)', 'Voltage(V)', 'Current(A)', 'Charge_Capacity(Ah)']
    for col in required:
        if col not in df.columns:
            return None

    df = df.copy().sort_values('Test_Time(s)')

    temp_cols = find_temperature_cols(df)
    if temp_cols:
        df['temperature_mean'] = df[temp_cols].mean(axis=1)
    else:
        df['temperature_mean'] = np.nan

    charge = df[df['Current(A)'] > 0.05].copy()

    if len(charge) < 10:
        return None

    t_start = charge['Test_Time(s)'].iloc[0]
    t_end = charge['Test_Time(s)'].iloc[-1]

    last_60 = charge[charge['Test_Time(s)'] >= t_end - 60]
    last_300 = charge[charge['Test_Time(s)'] >= t_end - 300]

    if len(last_60) < 2:
        last_60 = charge.tail(min(10, len(charge)))

    voltage_slope_last_60 = np.polyfit(
        last_60['Test_Time(s)'],
        last_60['Voltage(V)'],
        1
    )[0] if len(last_60) >= 2 else np.nan

    current_slope_last_60 = np.polyfit(
        last_60['Test_Time(s)'],
        last_60['Current(A)'],
        1
    )[0] if len(last_60) >= 2 else np.nan

    features = {
        'charge_duration': t_end - t_start,
        'voltage_start': charge['Voltage(V)'].iloc[0],
        'voltage_end': charge['Voltage(V)'].iloc[-1],
        'voltage_mean': charge['Voltage(V)'].mean(),
        'voltage_mean_last_60s': last_60['Voltage(V)'].mean(),
        'voltage_std_last_60s': last_60['Voltage(V)'].std(),
        'voltage_slope_last_60s': voltage_slope_last_60,
        'current_start': charge['Current(A)'].iloc[0],
        'current_end': charge['Current(A)'].iloc[-1],
        'current_mean': charge['Current(A)'].mean(),
        'current_mean_last_60s': last_60['Current(A)'].mean(),
        'current_std_last_60s': last_60['Current(A)'].std(),
        'current_slope_last_60s': current_slope_last_60,
        'charge_capacity_end': charge['Charge_Capacity(Ah)'].iloc[-1],
        'temperature_end': charge['temperature_mean'].iloc[-1],
        'temperature_max': charge['temperature_mean'].max(),
        'temperature_rise': charge['temperature_mean'].iloc[-1] - charge['temperature_mean'].iloc[0],
        'n_charge_points': len(charge),
        'n_last_60_points': len(last_60),
        'n_last_300_points': len(last_300),
    }

    if 'dV/dt(V/s)' in charge.columns:
        features['dvdt_mean_last_60s'] = last_60['dV/dt(V/s)'].mean()
        features['dvdt_std_last_60s'] = last_60['dV/dt(V/s)'].std()

    return features

In [ ]:
cycling_files = list((data_dir / 'cycling_tests').rglob('*.xlsx'))

feature_rows = []
skipped_features = []

for i, file in enumerate(cycling_files):
    if i % 20 == 0:
        print(i, '/', len(cycling_files), file.name)

    cell_id = parse_cell_id(file)
    cycling_id = parse_cycling_id(file)

    if cell_id is None or cycling_id is None:
        skipped_features.append((file, 'parse failed'))
        continue

    try:
        df = read_channel_sheet(file)
        if df is None:
            skipped_features.append((file, 'no channel sheet'))
            continue

        feats = extract_post_charge_features(df)
        if feats is None:
            skipped_features.append((file, 'feature failed'))
            continue

        feats.update({
            'cell_id': cell_id,
            'cycling_id': cycling_id,
            'file': str(file)
        })

        feature_rows.append(feats)

    except Exception as e:
        skipped_features.append((file, str(e)))

features_raw = pd.DataFrame(feature_rows)

print('features_raw:', features_raw.shape)
print('skipped:', len(skipped_features))
features_raw.head()

In [ ]:
feature_cols = [
    c for c in features_raw.columns
    if c not in ['cell_id', 'cycling_id', 'file']
]

features = (
    features_raw
    .groupby(['cell_id', 'cycling_id'], as_index=False)[feature_cols]
    .mean()
    .sort_values(['cell_id', 'cycling_id'])
    .reset_index(drop=True)
)

features['rpt_id'] = features['cycling_id'] + 1

dataset = features.merge(
    labels[['cell_id', 'rpt_id', 'soh', 'measured_capacity_ah']],
    on=['cell_id', 'rpt_id'],
    how='inner'
)

print(features.shape)
print(dataset.shape)
dataset.head()

In [ ]:
features.to_csv('/content/drive/MyDrive/battery_data/stanford_post_charge_features_only.csv', index=False)
dataset.to_csv('/content/drive/MyDrive/battery_data/stanford_post_charge_features.csv', index=False)